# SafeLink CNN — Full Retrain Guide

## Before you run this notebook

This notebook needs **Python 3.12** (TensorFlow does not support Python 3.14).

**One-time setup in your terminal:**
```
# 1. Download & install Python 3.12 from https://www.python.org/downloads/release/python-3129/
#    (tick 'Add to PATH' during install)

# 2. Open terminal inside ml_pipeline/ folder, then:
py -3.12 -m venv venv_train
venv_train\Scripts\activate

# 3. Install everything
pip install tensorflow pandas scikit-learn tqdm numpy requests jupyter ipywidgets

# 4. Open this notebook
jupyter notebook retrain.ipynb
```

Then run **all cells top to bottom**. Each cell prints its own progress.

---
**What this notebook does:**
1. Checks your environment
2. Downloads the 3 missing datasets (Tranco, URLhaus, OpenPhish)
3. Builds `safelink_dataset.csv` from all 5 sources
4. Trains the CNN+Dense model → `safelink_model.tflite`
5. Trains the Autoencoder → `autoencoder.tflite`
6. Copies all assets to the Android `assets/` folder

In [22]:
# ── Cell 1: Environment check ──────────────────────────────────────
import sys, os

# Pin working directory to ml_pipeline/ so all relative paths work correctly
try:
    _nb_dir = os.path.dirname(__vsc_ipynb_file__)   # VS Code sets this
except NameError:
    _nb_dir = os.path.dirname(os.path.abspath('retrain.ipynb'))
os.chdir(_nb_dir)
print('Working dir:', os.getcwd())

print('Python:', sys.version)

major, minor = sys.version_info.major, sys.version_info.minor
if major != 3 or minor not in range(8, 13):
    print()
    print('ERROR: TensorFlow requires Python 3.8–3.12.')
    print(f'       You are running Python {major}.{minor}.')
    print('       Follow the setup steps in the markdown cell above.')
    raise SystemExit('Wrong Python version — see instructions above.')

try:
    import tensorflow as tf
    print('TensorFlow:', tf.__version__)
except ImportError:
    raise SystemExit('TensorFlow not found. Run: pip install tensorflow')

import pandas as pd, numpy as np, sklearn, requests
print('pandas:', pd.__version__)
print('numpy:', np.__version__)
print('sklearn:', sklearn.__version__)
print()
print('All checks passed — ready to retrain.')

Working dir: g:\BICT\Project\SafeLink\ml_pipeline
Python: 3.12.0 (tags/v3.12.0:0fb18b0, Oct  2 2023, 13:03:39) [MSC v.1935 64 bit (AMD64)]
TensorFlow: 2.21.0
pandas: 3.0.2
numpy: 2.4.4
sklearn: 1.8.0

All checks passed — ready to retrain.


In [23]:
# ── Cell 2: Download Tranco top-1M (benign domains) ───────────────
# Tranco is a research-grade list of the top 1 million legitimate domains.
# Format: rank,domain  (no header row)
# We take only the top 10,000 rows (enough signal, manageable size).

import requests, zipfile, io, os

TRANCO_OUT = '../Datasets/tranco_top1m.csv'

if os.path.exists(TRANCO_OUT):
    lines = sum(1 for _ in open(TRANCO_OUT))
    print(f'[SKIP] tranco_top1m.csv already exists ({lines:,} rows)')
else:
    print('[DOWNLOAD] Tranco top-1M ...')
    # Permanent URL for the latest Tranco top-1M list
    r = requests.get('https://tranco-list.eu/top-1m.csv.zip',
                     timeout=120, allow_redirects=True)
    r.raise_for_status()
    print(f'  Downloaded {len(r.content)/1024:.0f} KB')

    # Unzip and write
    z = zipfile.ZipFile(io.BytesIO(r.content))
    inner_name = z.namelist()[0]   # usually 'top-1m.csv'
    print(f'  File inside zip: {inner_name}')
    with z.open(inner_name) as zf:
        content = zf.read().decode('utf-8')
    with open(TRANCO_OUT, 'w', encoding='utf-8') as f:
        f.write(content)
    lines = content.count('\n')
    print(f'[OK] Saved tranco_top1m.csv  ({lines:,} rows)')

[SKIP] tranco_top1m.csv already exists (2,000,000 rows)


In [24]:
# ── Cell 3: Download URLhaus (malware URLs) ────────────────────────
# URLhaus is abuse.ch's database of malware delivery URLs.
# The CSV has comment lines starting with '#' and a 'url' column.

import requests, zipfile, io, os

URLHAUS_OUT = '../Datasets/urlhaus.csv'

if os.path.exists(URLHAUS_OUT):
    lines = sum(1 for _ in open(URLHAUS_OUT, encoding='utf-8', errors='replace'))
    print(f'[SKIP] urlhaus.csv already exists ({lines:,} lines)')
else:
    print('[DOWNLOAD] URLhaus CSV (may be ~5MB zipped) ...')
    # Full URLhaus dataset — zip containing csv.txt
    r = requests.get('https://urlhaus.abuse.ch/downloads/csv/',
                     timeout=120, allow_redirects=True)
    r.raise_for_status()
    print(f'  Downloaded {len(r.content)/1024:.0f} KB')

    z = zipfile.ZipFile(io.BytesIO(r.content))
    inner_name = z.namelist()[0]
    print(f'  File inside zip: {inner_name}')
    with z.open(inner_name) as zf:
        content = zf.read().decode('utf-8', errors='replace')
    with open(URLHAUS_OUT, 'w', encoding='utf-8') as f:
        f.write(content)
    lines = content.count('\n')
    print(f'[OK] Saved urlhaus.csv  ({lines:,} lines)')

[SKIP] urlhaus.csv already exists (152,056 lines)


In [25]:
# ── Cell 4: Download OpenPhish (active phishing URLs) ─────────────
# OpenPhish free tier: plain text feed, one URL per line.
# No account needed.

import requests, os

OPENPHISH_OUT = '../Datasets/openphish.txt'

if os.path.exists(OPENPHISH_OUT):
    lines = sum(1 for _ in open(OPENPHISH_OUT))
    print(f'[SKIP] openphish.txt already exists ({lines:,} URLs)')
else:
    print('[DOWNLOAD] OpenPhish feed ...')
    r = requests.get('https://openphish.com/feed.txt', timeout=60)
    r.raise_for_status()
    with open(OPENPHISH_OUT, 'w', encoding='utf-8') as f:
        f.write(r.text)
    lines = r.text.count('\n')
    print(f'[OK] Saved openphish.txt  ({lines:,} URLs)')

[SKIP] openphish.txt already exists (300 URLs)


In [26]:
# ── Cell 5: Download PhishTank / PhishStats (phishing URLs) ───────
# PhishTank requires a free API key (register at phishtank.com).
# Set env var PHISHTANK_KEY=<your_key> to use it.
# Without a key this cell auto-downloads from PhishStats (free, no auth).

import requests, os, io
import pandas as pd

PHISHTANK_OUT = '../Datasets/phishtank.csv'

if os.path.exists(PHISHTANK_OUT):
    import csv
    with open(PHISHTANK_OUT) as f:
        rows = sum(1 for _ in csv.reader(f))
    print(f'[SKIP] phishtank.csv already exists ({rows:,} rows)')
else:
    phishtank_key = os.environ.get('PHISHTANK_KEY', '').strip()

    if phishtank_key:
        # ── PhishTank authenticated download ──────────────────────
        print('[DOWNLOAD] PhishTank (authenticated) ...')
        url = f'https://data.phishtank.com/data/{phishtank_key}/online-valid.csv.gz'
        r = requests.get(url, timeout=120,
                         headers={'User-Agent': 'phishtank/SafeLink'})
        r.raise_for_status()
        df = pd.read_csv(io.BytesIO(r.content), compression='gzip',
                         usecols=['url'], low_memory=False)
        df.to_csv(PHISHTANK_OUT, index=False)
        print(f'[OK] Saved phishtank.csv  ({len(df):,} rows from PhishTank)')

    else:
        # ── PhishStats fallback (free, no auth needed) ─────────────
        print('[DOWNLOAD] PhishStats feed (PHISHTANK_KEY not set — using free fallback) ...')
        urls = []
        per_page = 500
        for page in range(20):   # up to 10,000 URLs
            try:
                r = requests.get(
                    'https://phishstats.info:2096/api/phishing',
                    params={'_where': '(score,gte,5)', '_size': per_page, '_p': page},
                    timeout=30
                )
                if r.status_code != 200:
                    break
                batch = r.json()
                if not batch:
                    break
                for entry in batch:
                    u = entry.get('url')
                    if u and isinstance(u, str) and u.startswith('http'):
                        urls.append(u)
                if len(batch) < per_page:
                    break
            except Exception as e:
                print(f'  [WARN] page {page} failed: {e}')
                break

        df = pd.DataFrame({'url': list(dict.fromkeys(urls))})  # deduplicate, preserve order
        df.to_csv(PHISHTANK_OUT, index=False)
        print(f'[OK] Saved phishtank.csv  ({len(df):,} rows from PhishStats)')

[DOWNLOAD] PhishStats feed (PHISHTANK_KEY not set — using free fallback) ...
[OK] Saved phishtank.csv  (0 rows from PhishStats)


In [27]:
# ── Cell 6: Verify all datasets ────────────────────────────────────
import os

DATASETS = {
    '../Datasets/malicious_phish 2021 .csv':        'malicious_phish_2021  (required)',
    '../Datasets/PhiUSIIL_Phishing_URL_Dataset.csv': 'PhiUSIIL              (required)',
    '../Datasets/tranco_top1m.csv':                  'Tranco                (just downloaded)',
    '../Datasets/urlhaus.csv':                       'URLhaus               (just downloaded)',
    '../Datasets/openphish.txt':                     'OpenPhish             (just downloaded)',
    '../Datasets/phishtank.csv':                     'PhishTank             (optional)',
}

print('Dataset status:')
print('-' * 60)
missing_required = []
for path, label in DATASETS.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f'  [OK]   {label}  ({size_mb:.1f} MB)')
    else:
        print(f'  [MISS] {label}')
        if 'required' in label:
            missing_required.append(path)

if missing_required:
    raise RuntimeError(f'Required datasets missing: {missing_required}')
else:
    print()
    print('All required datasets present. Ready to build training data.')

Dataset status:
------------------------------------------------------------
  [OK]   malicious_phish_2021  (required)  (34.4 MB)
  [OK]   PhiUSIIL              (required)  (51.7 MB)
  [OK]   Tranco                (just downloaded)  (22.6 MB)
  [OK]   URLhaus               (just downloaded)  (15.1 MB)
  [OK]   OpenPhish             (just downloaded)  (0.0 MB)
  [OK]   PhishTank             (optional)  (0.0 MB)

All required datasets present. Ready to build training data.


In [28]:
# ── Cell 7: Build safelink_dataset.csv ────────────────────────────
# Runs data_collector.py — merges all sources, extracts 36 features,
# saves to data/safelink_dataset.csv.
# This will take 5–15 minutes depending on your CPU.

print('Building safelink_dataset.csv ...')
print('(This runs feature extraction on all URLs in parallel — takes a few minutes)\n')

%run data_collector.py

Building safelink_dataset.csv ...
(This runs feature extraction on all URLs in parallel — takes a few minutes)

[LOAD] malicious_phish_2021: 554,734 rows after dropping defacement
[LOAD] PhiUSIIL: 235,795 rows
[LOAD] PhishTank+OpenPhish: 300 rows
[LOAD] Tranco: 10,000 rows
[LOAD] URLhaus: 76,019 rows

[MERGE] Combined: 876,848 rows
[DEDUP] 9,407 duplicates removed -> 867,438 rows

[FEATURES] Extracting 36 features from 867,438 URLs using 8 workers...


Chunks: 100%|██████████| 174/174 [01:00<00:00,  2.89it/s]



[DISTRIBUTION]
  SAFE (0):         572,927
  MALICIOUS (2):    294,511
  Total:            867,438

[DONE] Saved 867,438 rows -> data\safelink_dataset.csv
       Columns: ['url', 'label', 'url_length', 'domain_length', 'path_length', 'query_length', 'num_dots', 'num_hyphens', 'num_underscores', 'num_slashes', 'num_at_signs', 'num_question_marks', 'num_equals', 'num_ampersands', 'num_percent_signs', 'num_digits', 'digit_ratio', 'url_entropy', 'domain_entropy', 'is_https', 'is_ip_address', 'num_subdomains', 'has_suspicious_tld', 'has_trusted_tld', 'phishing_keyword_count', 'brand_keyword_count', 'has_url_shortener', 'num_query_params', 'has_port_in_url', 'has_double_slash_in_path', 'subdomain_length', 'sld_length', 'num_special_chars_in_domain', 'path_entropy', 'has_hex_encoding', 'consecutive_digits_in_domain', 'has_tld_in_path', 'domain_age_days']


In [29]:
# ── Cell 8: Dataset statistics ─────────────────────────────────────
import pandas as pd

df = pd.read_csv('data/safelink_dataset.csv', low_memory=False)

print(f'Total rows  : {len(df):,}')
print(f'Columns     : {len(df.columns)}')
print()
print('Label distribution:')
dist = df['label'].value_counts().sort_index()
labels = {0: 'SAFE (0)', 2: 'MALICIOUS (2)'}
for lbl, count in dist.items():
    pct = count / len(df) * 100
    print(f'  {labels.get(lbl, lbl):<20}: {count:>10,}  ({pct:.1f}%)')

print()
print('Source breakdown:')
if 'source' in df.columns:
    print(df['source'].value_counts().to_string())

print()
print('Feature sample (first 5 features, 3 URLs):')
from feature_extractor import FEATURE_COLUMNS
print(df[['url'] + FEATURE_COLUMNS[:5]].sample(3, random_state=1).to_string(index=False))

Total rows  : 867,438
Columns     : 38

Label distribution:
  SAFE (0)            :    572,927  (66.0%)
  MALICIOUS (2)       :    294,511  (34.0%)

Source breakdown:

Feature sample (first 5 features, 3 URLs):
                                                                                                 url  url_length  domain_length  path_length  query_length  num_dots
                                                        lyricsmode.com/lyrics/a/arashi/pikanchi.html        44.0           14.0         30.0           0.0       2.0
docstoc.com/docs/47036341/Esthetique-de-lhorreur--Le-genocide-rwandais-dans-la-litterature-africaine       100.0           11.0         89.0           0.0       1.0
                                                                         unmuseum.org/flystrange.htm        27.0           12.0         15.0           0.0       2.0


In [30]:
# ── Cell 9: Train CNN + Dense model ───────────────────────────────
# Architecture: Character CNN stream + 36-feature Dense stream → 3-class softmax
# Output: models/safelink_model.tflite + models/scaler_params.json
# Expected accuracy: ~97–99% on test set
# Expected time: 5–20 min on CPU depending on dataset size

print('Training CNN model ...')
print('Watch val_accuracy — should reach >0.97 within a few epochs.\n')

%run trainer.py

Training CNN model ...
Watch val_accuracy — should reach >0.97 within a few epochs.

[LOAD] Reading data/safelink_dataset.csv ...
       867,438 rows, columns: ['url', 'label', 'url_length', 'domain_length', 'path_length'] ...
[FILTER] After keeping SAFE+MALICIOUS: 867,438 rows
[ENCODE] Tokenizing URLs ...
[SPLIT] Train: 693,950  Val: 86,744  Test: 86,744
[SCALER] Fitting StandardScaler on training features ...
[SCALER] Saved -> models\scaler_params.json  (n_features=36)


c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\layer.py:1035: UserWarning: Layer 'conv1d' (of type Conv1D) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ seq_input           │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 32)   │      3,232 │ seq_input[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 200, 64)   │      6,208 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ num_input           │ (None, 36)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64)        │      2,368 │ num_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      6,208 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 3)         │        195 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 24,451 (95.51 KB)

...

c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(a

INFO:tensorflow:Assets written to: C:\Users\Shuhad\AppData\Local\Temp\tmphshifygk\assets


INFO:tensorflow:Assets written to: C:\Users\Shuhad\AppData\Local\Temp\tmphshifygk\assets


Saved artifact at 'C:\Users\Shuhad\AppData\Local\Temp\tmphshifygk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 200), dtype=tf.int32, name='seq_input'), TensorSpec(shape=(None, 36), dtype=tf.float32, name='num_input')]
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  1728901348624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728901348432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728901348816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009959376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009961104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009961296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009961488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728901348048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009958992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1729009963600

c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [31]:
# ── Cell 10: Train Autoencoder (anomaly detector) ──────────────────
# Trained on BENIGN URLs only. At inference, high reconstruction MSE = anomaly.
# Output: models/autoencoder.tflite + models/anomaly_threshold.json
# Expected time: 3–10 min on CPU

print('Training Autoencoder ...')
print('Trained on SAFE URLs only — val_loss should steadily decrease.\n')

%run autoencoder.py

Training Autoencoder ...
Trained on SAFE URLs only — val_loss should steadily decrease.

[LOAD] Reading data/safelink_dataset.csv ...
[FILTER] Benign URLs: 572,927
[SPLIT] Train: 515,634  Val: 57,293


Model: "autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ ae_input (InputLayer)           │ (None, 36)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 24)             │           888 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 12)             │           300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 24)             │           312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ ae_output (Dense)               │ (None, 36)             │           900 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,400 (9.38 KB)

 Trainable params: 2,400 (9.38 KB)

...

INFO:tensorflow:Assets written to: C:\Users\Shuhad\AppData\Local\Temp\tmprff81hsn\assets


Saved artifact at 'C:\Users\Shuhad\AppData\Local\Temp\tmprff81hsn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 36), dtype=tf.float32, name='ae_input')
Output Type:
  TensorSpec(shape=(None, 36), dtype=tf.float32, name=None)
Captures:
  1728963841488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963842256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963843216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963842448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963841872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963843792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963842064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1728963844368: TensorSpec(shape=(), dtype=tf.resource, name=None)


c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


[TFLITE] Saved -> models\autoencoder.tflite  (8.0 KB)
[OK]    Within 300KB target

[VALIDATE] TFLite autoencoder check ...
  Input:  serving_default_ae_input:0  shape=[ 1 36]  dtype=<class 'numpy.float32'>
  Output: StatefulPartitionedCall_1:0  shape=[ 1 36]  dtype=<class 'numpy.float32'>
  Sample MSE: 0.005350  Category: short  Threshold: 0.031453  Anomaly: False

[DONE] Autoencoder training complete.


c:\Users\Shuhad\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [32]:
# ── Cell 11: Verify model outputs ─────────────────────────────────
import os, struct

MODELS = {
    'models/safelink_model.tflite':  (20*1024,  300*1024),   # 20–300 KB
    'models/autoencoder.tflite':     (5*1024,   200*1024),   # 5–200 KB
    'models/scaler_params.json':     (500,      50*1024),    # 0.5–50 KB
    'models/anomaly_threshold.json': (10,       1024),       # tiny JSON
    'models/urlbert.onnx':           (10*1024*1024, 30*1024*1024),  # 10–30 MB
}

print('Model file verification:')
print('-' * 55)
all_ok = True
for path, (min_b, max_b) in MODELS.items():
    if not os.path.exists(path):
        print(f'  [MISS]  {path}')
        all_ok = False
        continue
    size = os.path.getsize(path)
    size_str = f'{size/1024:.1f} KB' if size < 1024*1024 else f'{size/1024/1024:.1f} MB'
    # Validate TFLite magic for .tflite files
    if path.endswith('.tflite'):
        with open(path, 'rb') as f:
            f.seek(4); magic = f.read(4)
        magic_ok = magic == b'TFL3'
        mark = '[OK]  ' if magic_ok and min_b <= size <= max_b else '[WARN]'
        print(f'  {mark}  {path}  ({size_str})  magic={magic}')
    else:
        mark = '[OK]  ' if min_b <= size <= max_b else '[WARN]'
        print(f'  {mark}  {path}  ({size_str})')

print()
if all_ok:
    print('All model files present. Proceed to export.')
else:
    print('Some files missing — re-run the training cells above.')

Model file verification:
-------------------------------------------------------
  [OK]    models/safelink_model.tflite  (35.0 KB)  magic=b'TFL3'
  [OK]    models/autoencoder.tflite  (8.0 KB)  magic=b'TFL3'
  [OK]    models/scaler_params.json  (2.7 KB)
  [OK]    models/anomaly_threshold.json  (0.2 KB)
  [OK]    models/urlbert.onnx  (14.9 MB)

All model files present. Proceed to export.


In [33]:
# ── Cell 12: Export assets to Android ─────────────────────────────
# Copies all model files into android/app/src/main/assets/
# This is what the Android app reads at runtime.

%run export_assets.py

SafeLink Asset Export & Validation
  [COPY] safelink_model.tflite  (35.0 KB)  -> ../android/app/src/main/assets\safelink_model.tflite
  [COPY] autoencoder.tflite  (8.0 KB)  -> ../android/app/src/main/assets\autoencoder.tflite
  [SKIP] urlbert.tflite (optional, not found)
  [COPY] urlbert.onnx  (15300.8 KB)  -> ../android/app/src/main/assets\urlbert.onnx
  [OK] scaler n_features=36, mean[0]=48.8423
  [COPY] scaler_params.json  (2.7 KB)  -> ../android/app/src/main/assets\scaler_params.json
  [OK] thresholds: short=0.031453, medium=0.059290, long=0.497741
  [COPY] anomaly_threshold.json  (0.2 KB)  -> ../android/app/src/main/assets\anomaly_threshold.json
  [SKIP] urlbert_meta.json (optional, not found)

WARNINGS:
  [!]  urlbert.onnx: 15300.8 KB exceeds 6144 KB target

[OK] All required assets exported successfully.
     Destination: g:\BICT\Project\SafeLink\android\app\src\main\assets

Assets in Android assets/:
  anomaly_threshold.json                   0.2 KB
  attack_patterns.json      

In [34]:
# ── Cell 13: Final verification ────────────────────────────────────
import os

ASSETS_DIR = '../android/app/src/main/assets'
REQUIRED_ASSETS = [
    'safelink_model.tflite',
    'autoencoder.tflite',
    'scaler_params.json',
    'anomaly_threshold.json',
    'urlbert.onnx',
    'vocab.txt',
]

print('Android assets check:')
print('-' * 50)
all_ready = True
for fname in REQUIRED_ASSETS:
    fpath = os.path.join(ASSETS_DIR, fname)
    if os.path.exists(fpath):
        size = os.path.getsize(fpath)
        size_str = f'{size/1024:.1f} KB' if size < 1024*1024 else f'{size/1024/1024:.1f} MB'
        print(f'  [OK]   {fname:<35} {size_str}')
    else:
        print(f'  [MISS] {fname}')
        all_ready = False

print()
if all_ready:
    print('All assets in place.')
    print('Next: Open Android Studio, sync Gradle, and build the APK.')
else:
    print('Some assets missing — check export_assets.py output above.')

Android assets check:
--------------------------------------------------
  [OK]   safelink_model.tflite               35.0 KB
  [OK]   autoencoder.tflite                  8.0 KB
  [OK]   scaler_params.json                  2.7 KB
  [OK]   anomaly_threshold.json              0.2 KB
  [OK]   urlbert.onnx                        14.9 MB
  [OK]   vocab.txt                           2.0 KB

All assets in place.
Next: Open Android Studio, sync Gradle, and build the APK.
